In [1]:
!pip install -q google-generativeai

import google.generativeai as genai
from getpass import getpass

# getpass keeps the key out of notebook output — important if you share the .ipynb
genai.configure(api_key=getpass("Paste your Gemini API key: "))

# Verify the key works before doing anything else
print(genai.GenerativeModel("gemma-4-26b-a4b-it").generate_content("Say OK").text.strip())

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Paste your Gemini API key: ··········
The user is asking me to "Say OK".
The instruction is extremely simple and direct.
"OK" (or "ok", "Okay", etc.).
The user wants a confirmation of their command.

Plan:
1. Respond with the word "OK".
OK


Cell A — Load the labelled dataset


In [2]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
df = pd.read_excel(list(uploaded.keys())[0])

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(3)

Saving Labeled dataset.xlsx to Labeled dataset.xlsx
Rows: 300
Columns: ['Abstract', 'Manual Classification']


,Abstract,Manual Classification
0,We study empirical scaling laws for language m...,Deep Learning
1,The reliability of artificial intelligence hin...,Machine Learning
2,"We trained a large, deep convolutional neural ...",Deep Learning


Cell B — Config, blocks, prompts, parsers

In [4]:
import google.generativeai as genai
import re, time, os
import pandas as pd

# ============================================================
# COLUMN NAMES — must match the sheet exactly
# ============================================================
ABSTRACT_COL = "Abstract"
MANUAL_COL   = "Manual Classification"
ID_COL       = None          # None = use the dataframe row index as paper_id

# ============================================================
# RUN CONFIG
# ============================================================
MODEL      = "gemma-4-26b-a4b-it"   # exact tag, logged per row (version drift)
OUTFILE    = "results_gemini_labelled.csv"
CATEGORIES = ["AI", "ML", "DL", "Unclear", "None"]


# ============================================================
# CATEGORY BLOCK
# Identical across ALL versions (V1, V2.1, V2.2, V3) so that the
# only thing changing between V2.1 and V2.2 is the RULES block.
# ============================================================
CAT_BLOCK = """- AI: general artificial intelligence approaches, symbolic reasoning, expert systems, or work not specific to ML/DL techniques
- ML: classical machine learning — supervised/unsupervised learning, tree-based methods, SVMs, clustering, feature engineering
- DL: neural network architectures — CNNs, RNNs, transformers, deep reinforcement learning
- Unclear: the abstract does not provide enough information to assign a category individually. Like there is a mix of multiple categories in the abstract so unable to assign individual categories
- None: the paper is not about AI, ML or DL at all — these terms appear only incidentally or as background, and the paper's contribution lies in another field entirely"""


# ============================================================
# RULES — OLD (initial version, used by V2.1)
# ============================================================
RULES_OLD = """Disambiguation rules:
- If the paper's core method is a neural network, classify as DL even though it is also ML and AI
- If multiple methods are compared, classify by the paper's primary contribution
- Use Unclear only when the abstract genuinely lacks the information needed, not when the paper is merely interdisciplinary
- Use None when AI/ML/DL are mentioned only as enabling technology or future possibility, and the paper itself contributes to a different field"""


# ============================================================
# RULES — NEW (current version, used by V2.2 and V3)
# Difference from OLD: the added "primary approach" rule below.
# ============================================================
RULES_NEW = RULES_OLD + """
- The context behind the classification is to identify the primary approach of the classification. Which is done by identifying the core concept the "methodology", "technical implementation" or the "review approach" of the paper follows to achieve its results, inference or conclusion"""


# ============================================================
# FEW-SHOT EXAMPLES (V3 only)
# Formatted in the same REASONING/CATEGORY shape the model must output,
# so the demonstrations reinforce the required response format.
# Ordered None -> AI -> ML -> DL -> Unclear.
# ============================================================
FEWSHOT_BLOCK = """Worked examples:

--- EXAMPLE 1 ---
Abstract: Artificial intelligence offers great opportunities in critical care, particularly when a vast amount of continuously acquired physiological data is incorporated. High-quality, reliably labelled data are paramount for developing and training artificial intelligence methods. However, routinely recorded data in critical care are often noisy, and the sheer volume of high-resolution data is challenging to manage. Generalizable solutions for these problems are lacking, restricting progress. To address these barriers, we developed Vitabel, an open-source Python framework for post hoc loading, visualizing, aligning, and annotating medical time series. The framework provides sensible defaults and interactive components for efficient use in preconfigured workflows, while remaining flexible and extendable for custom analysis and annotation pipelines. It integrates seamlessly into Jupyter Notebooks, providing an interactive, customizable interface for visual interaction with the data. In this publication, we demonstrate its utility across three use cases. The code and exemplary data are provided as browser-based demos. Vitabel is freely available and published under the MIT license accompanying this publication.
REASONING: The paper's contribution is a Python framework for loading, visualising and annotating medical time series. Although AI is mentioned as motivation, no AI, ML or DL concept, method or architecture is developed or studied.
CATEGORY: None

--- EXAMPLE 2 ---
Abstract: Background: The integration of artificial intelligence (AI) into traditional Chinese medicine (TCM) research and development offers promising solutions to longstanding challenges in the field. These challenges include the complexity of TCM formulations, variability in quality control, and hurdles in global market acceptance. The unique synergy between AI technologies and TCM principles creates opportunities to enhance research efficiency, standardization, and innovation. Aim of review: This review aims to explore the applications and impact of AI across three critical stages of TCM development: drug design, pharmaceutical manufacturing, and market access. By summarizing the advancements and limitations in these areas, the review identifies the transformative potential of AI and proposes future directions for integrating AI with emerging technologies to advance TCM research and development (R&D). Key scientific concepts of review: AI has transformative potential in TCM development, addressing key challenges across various stages. In drug design, AI accelerates the identification of active compounds, optimizes formula composition, and models pharmacodynamic relationships to enhance innovation efficiency and precision. During pharmaceutical manufacturing, AI contributes to process optimization, quality control, and the standardization of TCM products, ensuring stable and scalable production. For market access, although no TCM developed by AI has entered the clinic, AI has played a role in comprehensive safety and efficacy assessments and simplified regulatory compliance in other drugs. By leveraging these advances and reviewing limitations, AI promotes the need to develop more integrated, more efficient, and more utilized methods in TCM R&D.
REASONING: The review discusses AI across TCM drug design, manufacturing, and market access without naming a single algorithm, architecture, or learning methodology anywhere. The treatment of AI remains at the level of general capability rather than any specific ML or DL technique.
CATEGORY: AI

--- EXAMPLE 3 ---
Abstract: Background: Understanding how the respiratory microbiota matures with age is key to improving poultry health and pathogen surveillance, yet the ecological processes shaping this transition remain elusive. We aimed to develop an interpretable machine-learning framework capable of identifying age-associated microbial signatures within the chicken nasal microbiota across heterogeneous datasets. Results: We compiled data from five independent chicken studies and normalized microbial abundances using Counts Per Million (CPM). To address dataset imbalance and ensure cross-study generalizability, we implemented SMOTE over-sampling and a Leave-One-Study-Out (LOSO) cross-validation framework. Within this architecture, we utilized Recursive Feature Elimination (RFE) to identify a stable consensus signature composed of taxa persisting in at least 70% of the iterations. We benchmarked five algorithms: Classification and Regression Trees (CART), k-nearest neighbors (kNN), Support Vector Machines (SVM), Random Forest (RF), and Extreme Gradient Boosting (XGBoost). RF emerged as the best model, achieving a balanced accuracy of 0.965 and a Kappa of 0.920. Consequently, the contribution of each feature was quantified through SHapley Additive exPlanations (SHAP) values on the selected RF model, enabling transparent interpretation of age-dependent microbial patterns. This approach distilled a compact set of predictive taxa, including Corynebacterium, Kocuria, and members of the Micrococcaceae. External validation with longitudinal samples from a Highly Pathogenic Avian Influenza Virus (HPAIV) infection confirmed full generalization, with all 57 samples from 22 chickens correctly classified even under viral-induced conditions. Conclusions: The proposed workflow combining LOSO-based feature selection, class-balancing, and interpretable machine learning provides a transferable framework for microbiota-based age inference.
REASONING: Every model in the paper is a classical algorithm of ML with no neural network anywhere, so it cannot sit at the deep learning tier. The rest of the work is feature engineering and validation design — normalisation, resampling, feature elimination, cross-validation — which is textbook machine learning practice. The poultry biology is just the subject matter the methods are applied to, and SHAP is only used afterwards to explain the chosen model.
CATEGORY: ML

--- EXAMPLE 4 ---
Abstract: Early detection and prediction of Depressive Symptoms is essential for improving mental health outcomes. This study proposes a hybrid deep learning and machine learning framework that utilizes tabular data collected from wearable devices, including sleep patterns, physical activity, and health-related indicators. Three ensemble learning models were used to identify influential predictors through the application of explainable artificial intelligence techniques such as SHAP and LIME. Based on the selected important features, two hybrid models were developed combining 1D-Convolutional Neural Network and Multi-layer Perceptron with LightGBM. The experimental results showed that models trained on selected features consistently outperformed those using the full feature set. The highest classification accuracy, 93.43%, was achieved by the multilayer perceptron model with LightGBM when trained on features selected by XGBoost. SHAP analysis highlighted the importance of features such as night sleep duration, age, and income responsibility, while LIME provided sample specific explanations that enhanced local interpretability. This framework enhances both the predictive performance and interpretability of depression prediction models and demonstrates the potential of wearable-derived behavior and physiological features as practical biomarkers for personalized risk assessment.
REASONING: The main subject of the paper involves neural network architecture and concepts like 1D-CNN and MLP. Although XGBoost and SHAP/LIME appear in the abstract, they are not the main subject of contribution — they are used only as tools supporting the aforementioned neural network architectures.
CATEGORY: DL

--- EXAMPLE 5 ---
Abstract: Artificial intelligence (AI) is reshaping education by enabling personalized learning and increasing student engagement. The rapid adoption of tools such as ChatGPT, however, raises questions about efficacy, academic integrity, and ethics. This study aims to fill a critical gap by providing a systematic comparative evaluation of classical, deep learning, and transformer-based models for sentiment analysis of ChatGPT-related educational discourse, identifying the best-performing approach, and deploying it in an explainable, user-friendly web application for non-technical stakeholders. This study gathered 236,275 tweets related to ChatGPT in education and implemented a systematic benchmarking process. A Naive Bayes model with TF-IDF vectorization served as the classical baseline. For the deep learning tier, Long Short-Term Memory (LSTM) and Bidirectional LSTM (Bi-LSTM) models with Word2Vec embeddings were used. The transformer tier was represented by a fine-tuned DistilBERT model. The fine-tuned DistilBERT achieved the highest accuracy at 98.81%. In contrast, the LSTM model achieved 95.40%, the Bi-LSTM reached 94.66%, and the Naive Bayes model recorded an accuracy of 81.77%. A Streamlit-based web application was developed that integrates LIME-based explainability.
REASONING: The study is layered across an AI system (ChatGPT) as its subject, classical ML (Naive Bayes, TF-IDF), and DL (LSTM, Bi-LSTM, DistilBERT), with each tier given comparable weight. No single category is identifiable as the primary contribution.
CATEGORY: Unclear"""

FEWSHOT_EXTRA_RULE = """- Use these examples to understand the context behind the classification. The primary approach or deciding factor of the classification is what core concept the "methodology", "technical implementation" or the "review approach" of the paper follows to achieve its results, inference or conclusion"""


# ============================================================
# COT INSTRUCTION BLOCK (shared by V2.1, V2.2, V3)
# ============================================================
COT_STEPS = """Work through this in three steps:
1. Identify the technical methods described in the text
2. Determine which method represents the paper's primary contribution
3. Assign the most specific applicable category

Respond in exactly this format:
REASONING: <two or three sentences>
CATEGORY: <one category name>"""

HEADER = """You are an expert in artificial intelligence and machine learning research.

IMPORTANT CONTEXT: These categories are hierarchically nested. Deep learning is a subset of machine learning, and machine learning is a subset of artificial intelligence. Assign the MOST SPECIFIC category that applies."""


# ============================================================
# PROMPT VERSIONS
#   V1   = definition-augmented zero-shot (no rules, no CoT)
#   V2.1 = CoT + OLD rules
#   V2.2 = CoT + NEW rules
#   V3   = CoT + few-shot + NEW rules
# ============================================================
PROMPTS = {

"V1": f"""You are classifying academic paper abstracts by their primary technical contribution.

Categories:
{CAT_BLOCK}

Assign the single category that best matches the paper's PRIMARY contribution.

Respond with only the category name.

{{text}}""",

"V2.1": f"""{HEADER}

Categories:
{CAT_BLOCK}

{RULES_OLD}

{COT_STEPS}

{{text}}""",

"V2.2": f"""{HEADER}

Categories:
{CAT_BLOCK}

{RULES_NEW}

{COT_STEPS}

{{text}}""",

"V3": f"""{HEADER}

Categories:
{CAT_BLOCK}

{RULES_NEW}
{FEWSHOT_EXTRA_RULE}

{FEWSHOT_BLOCK}

Now classify the following paper.

{COT_STEPS}

{{text}}""",
}

# Versions that emit REASONING/CATEGORY and therefore need CATEGORY-line parsing
COT_VERSIONS = {"V2.1", "V2.2", "V3"}


# ============================================================
# INPUT BUILDER — abstract only
# Title/keyword fallback removed. A blank abstract has no usable input
# and is logged as NO_INPUT rather than silently dropped.
# ============================================================
MISSING_TOKENS = {"", "nan", "none", "null", "n/a", "na"}

def _blank(v):
    """True if the cell is empty, NaN-like, or an explicit 'no abstract' marker."""
    s = str(v).strip().lower()
    return s in MISSING_TOKENS or "no abstract available" in s


def build_input(row):
    """Return (text_block, source_used). source_used: abstract | missing"""
    if not _blank(row.get(ABSTRACT_COL)):
        return f"Abstract: {str(row[ABSTRACT_COL]).strip()}", "abstract"
    return None, "missing"


# ============================================================
# LABEL NORMALISATION
# Manual labels use full words; model outputs use abbreviations.
# ============================================================
LABEL_MAP = {
    "artificial intelligence": "AI", "ai": "AI",
    "machine learning": "ML",        "ml": "ML",
    "deep learning": "DL",           "dl": "DL",
    "unclear": "Unclear",
    "none": "None",
}

def normalise(label):
    return LABEL_MAP.get(str(label).strip().lower(), str(label).strip())


# ============================================================
# OUTPUT PARSING
# Exact first-token match only. For CoT versions, parse ONLY what follows
# "CATEGORY:" so reasoning text cannot contaminate the label.
# ============================================================
def _match(token):
    token = token.strip(".,:;*\"'`")
    for c in CATEGORIES:
        if token.lower() == c.lower():
            return c
    return "UNPARSEABLE"


# Match a category only as a standalone word, not inside another word
_CAT_RE = re.compile(r"\b(AI|ML|DL|Unclear|None)\b", re.IGNORECASE)

def extract_label(raw, version):
    """1) Prefer an explicit CATEGORY: line (any position).
       2) Otherwise try the first token.
       3) Otherwise take the LAST standalone category mention — models that
          reason before answering put the verdict at the end."""
    text = raw.strip()

    # 1. Explicit CATEGORY: line — take the LAST one, since reasoning models
    #    sometimes restate the format before committing to an answer
    hits = re.findall(r"CATEGORY\s*:\s*\**\s*(\w+)", text, re.IGNORECASE)
    if hits:
        lab = _match(hits[-1])
        if lab != "UNPARSEABLE":
            return lab

    # 2. Clean first token (handles "AI", "**AI**", "AI.")
    stripped = text.strip("*# \n")
    parts = stripped.split()
    if parts:
        lab = _match(parts[0])
        if lab != "UNPARSEABLE":
            return lab

    # 3. Last standalone category word anywhere in the response
    found = _CAT_RE.findall(text)
    if found:
        canon = {"ai": "AI", "ml": "ML", "dl": "DL",
                 "unclear": "Unclear", "none": "None"}
        return canon[found[-1].lower()]

    return "UNPARSEABLE"


# ============================================================
# FEW-SHOT LEAKAGE TAGGING
# Examples remain in the evaluation set as instructed; these rows are
# simply tagged so V3 can be reported with and without them if needed.
# ============================================================
FEWSHOT_FINGERPRINTS = [
    "vitabel, an open-source python framework",
    "integration of artificial intelligence (ai) into traditional chinese medicine",
    "age-associated microbial signatures within the chicken nasal microbiota",
    "1d-convolutional neural network and multi-layer perceptron with lightgbm",
    "236,275 tweets related to chatgpt in education",
]

def is_fewshot_row(row):
    a = str(row.get(ABSTRACT_COL, "")).lower()
    return any(fp in a for fp in FEWSHOT_FINGERPRINTS)


# ============================================================
# PRE-RUN AUDIT
# ============================================================
for col in [ABSTRACT_COL, MANUAL_COL]:
    assert col in df.columns, f"Column '{col}' not found. Present: {list(df.columns)}"

sources = df.apply(lambda r: build_input(r)[1], axis=1)
src_counts = sources.value_counts()

print("=" * 55)
print(f"Rows: {len(df)}   Versions: {len(PROMPTS)}   Calls: {len(df) * len(PROMPTS)}")
print(f"Model: {MODEL}")
print("=" * 55)
print(f"\n  abstract usable ........... {src_counts.get('abstract', 0)}")
print(f"  no-abstract record count = {src_counts.get('missing', 0)}")

n_fs = df.apply(is_fewshot_row, axis=1).sum()
print(f"\nFew-shot example rows matched in dataset: {n_fs} / 5")
if n_fs < 5:
    print("  (unmatched examples are simply not in this sheet — not an error)")

print("\nManual label distribution:")
print(df[MANUAL_COL].map(normalise).value_counts())
print("=" * 55)

# ============================================================
# GEMINI CALLER
# Retries transient server errors (503/429) with exponential backoff.
# Returns (raw_text, error_string).
# ============================================================
TRANSIENT = ("503", "502", "500", "429", "overloaded", "unavailable",
             "quota", "rate", "timeout", "deadline", "internal")

def call_model(prompt, model_name, retries=6):
    model = genai.GenerativeModel(model_name)
    for attempt in range(retries):
        try:
            r = model.generate_content(
                prompt,
                generation_config={"temperature": 0, "max_output_tokens": 400}
            )
            txt = (r.text or "").strip()
            if not txt:
                return "", "empty response (possible safety block)"
            return txt, ""
        except Exception as e:
            msg = str(e)
            if any(t in msg.lower() for t in TRANSIENT):
                wait = min(4 * (2 ** attempt), 120)
                print(f"  transient error — retry {attempt+1}/{retries} in {wait}s", flush=True)
                time.sleep(wait)
                continue
            return "", msg
    return "", f"failed after {retries} retries"

Rows: 300   Versions: 4   Calls: 1200
Model: gemma-4-26b-a4b-it

  abstract usable ........... 297
  no-abstract record count = 3

Few-shot example rows matched in dataset: 2 / 5
  (unmatched examples are simply not in this sheet — not an error)

Manual label distribution:
Manual Classification
Unclear    106
ML          99
AI          52
DL          29
nan         14
Name: count, dtype: int64


In [5]:
import sys

# ---- Resume support: skip (paper, model, version) triples already on disk ----
if os.path.exists(OUTFILE):
    done_df = pd.read_csv(OUTFILE)
    done = set(zip(done_df["paper_id"].astype(str),
                   done_df["model_version"],
                   done_df["prompt_version"]))
    print(f"Resuming — {len(done)} rows already complete\n", flush=True)
else:
    done_df, done = pd.DataFrame(), set()

# ---- Fail fast if the API is unreachable ----
try:
    genai.GenerativeModel(MODEL).generate_content("test")
    print("✓ Gemini API reachable\n", flush=True)
except Exception as e:
    raise SystemExit(f"Gemini API unreachable — check the API key. ({e})")

rows, counter, skipped = [], 0, 0
total = len(df) * len(PROMPTS)
run_start = time.time()

print(f"{'#':>6} {'VER':<6} {'SRC':<9} {'MANUAL':<9} {'PRED':<12} {'SEC':>6} {'ETA':>7}")
print("-" * 66, flush=True)

for version, template in PROMPTS.items():
    for idx, r in df.iterrows():
        counter += 1
        pid = str(r[ID_COL]) if ID_COL else str(idx)

        if (pid, MODEL, version) in done:
            skipped += 1
            continue

        text_block, source = build_input(r)
        manual = normalise(r[MANUAL_COL])

        # Blank abstract — logged, not silently dropped
        if text_block is None:
            rows.append({
                "paper_id": pid, "model_version": MODEL, "prompt_version": version,
                "input_source": "missing", "manual_label": manual,
                "predicted_label": "NO_INPUT", "raw_output": "",
                "elapsed_sec": 0.0, "error": "no abstract available",
                "is_fewshot_example": is_fewshot_row(r),
                "run_timestamp": pd.Timestamp.now().isoformat(timespec="seconds"),
            })
            print(f"{counter:>6} {version:<6} {'missing':<9} {manual:<9} {'NO_INPUT':<12}", flush=True)
            continue

        # ---- API call (differs from the Ollama notebooks) ----
        t0 = time.time()
        raw, err = call_model(template.format(text=text_block), MODEL)
        elapsed = round(time.time() - t0, 2)

        time.sleep(1)   # stay inside the free-tier requests-per-minute cap

        label = extract_label(raw, version) if raw else "ERROR"

        rows.append({
            "paper_id":           pid,
            "model_version":      MODEL,
            "prompt_version":     version,
            "input_source":       source,
            "manual_label":       manual,
            "predicted_label":    label,
            "raw_output":         raw,          # kept for error analysis
            "elapsed_sec":        elapsed,      # NB: includes network latency
            "error":              err,
            "is_fewshot_example": is_fewshot_row(r),
            "run_timestamp":      pd.Timestamp.now().isoformat(timespec="seconds"),
        })

        # ---- live progress ----
        processed = counter - skipped
        avg = (time.time() - run_start) / max(processed, 1)
        eta = (total - counter) * avg / 60
        mark = "✓" if label == manual else ("✗" if label not in ("ERROR", "UNPARSEABLE") else "!")
        print(f"{counter:>6} {version:<6} {source:<9} {manual:<9} {label:<12} "
              f"{elapsed:>6.2f} {eta:>6.0f}m {mark}", flush=True)
        if err:
            print(f"       └─ ERROR: {err[:80]}", flush=True)

        # ---- flush every 10 rows so a disconnect costs little ----
        if len(rows) >= 10:
            pd.concat([done_df, pd.DataFrame(rows)], ignore_index=True).to_csv(OUTFILE, index=False)
            done_df = pd.read_csv(OUTFILE)
            rows = []

if rows:
    pd.concat([done_df, pd.DataFrame(rows)], ignore_index=True).to_csv(OUTFILE, index=False)

results = pd.read_csv(OUTFILE)
print("\n" + "=" * 66)
print(f"Complete — {len(results)} rows | {(time.time()-run_start)/60:.1f} min")
print(f"Errors: {results['error'].astype(str).ne('').sum()} | "
      f"Unparseable: {(results.predicted_label == 'UNPARSEABLE').sum()}", flush=True)

✓ Gemini API reachable

     # VER    SRC       MANUAL    PRED            SEC     ETA
------------------------------------------------------------------
     1 V1     abstract  DL        DL             8.04    181m ✓
     2 V1     abstract  ML        ML             8.89    189m ✓
     3 V1     abstract  DL        DL             7.11    180m ✓
     4 V1     abstract  AI        DL             8.99    185m ✗
     5 V1     abstract  DL        DL             8.81    187m ✓
     6 V1     abstract  ML        ML             9.04    189m ✓
     7 V1     abstract  Unclear   ML             8.87    190m ✗
     8 V1     abstract  ML        ML             6.65    185m ✓
     9 V1     abstract  Unclear   DL             8.92    186m ✗
    10 V1     abstract  ML        ML             8.94    187m ✓
    11 V1     abstract  Unclear   ML            10.75    191m ✗
    12 V1     abstract  DL        ML            75.55    301m ✗
    13 V1     abstract  ML        ML             8.92    293m ✓
    14 V1     a

ERROR:tornado.access:500 POST /v1beta/models/gemma-4-26b-a4b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 594646.45ms


    74 V1     abstract  Unclear   DL           629.01    373m ✗
    75 V1     abstract  DL        DL             8.90    370m ✓
    76 V1     abstract  ML        DL             9.02    367m ✗
    77 V1     abstract  Unclear   Unclear        8.94    364m ✓
    78 V1     abstract  ML        ML             9.42    362m ✓
    79 V1     abstract  ML        ML             9.24    359m ✓
    80 V1     abstract  ML        ML           102.17    379m ✓
    81 V1     abstract  DL        DL             7.86    376m ✓
    82 V1     abstract  ML        ML             8.92    373m ✓
    83 V1     abstract  ML        ML             7.51    370m ✓
    84 V1     abstract  AI        ML             9.04    368m ✗
    85 V1     abstract  Unclear   Unclear        8.87    365m ✓
    86 V1     abstract  DL        DL             9.00    363m ✓
    87 V1     abstract  Unclear   Unclear        8.79    360m ✓
    88 V1     abstract  ML        ML             8.57    358m ✓
    89 V1     abstract  DL        Unclea

  transient error — retry 1/6 in 4s
  transient error — retry 2/6 in 8s


   994 V3     abstract  Unclear   DL            21.74     81m ✗
   995 V3     abstract  Unclear   AI             9.17     80m ✗
   996 V3     abstract  AI        AI             9.17     80m ✓
   997 V3     abstract  ML        DL             9.17     80m ✗
   998 V3     abstract  AI        Unclear        8.99     79m ✗


...

  transient error — retry 2/6 in 8s


  1133 V3     abstract  nan       AI            21.58     25m ✗
  1134 V3     abstract  Unclear   ML             8.97     24m ✗
  1135 V3     abstract  ML        None           9.20     24m ✗
  1136 V3     abstract  ML        ERROR          0.01     24m !
       └─ ERROR: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without
  1137 V3     abstract  DL        DL             8.95     23m ✓
  1138 V3     abstract  AI        AI             8.84     23m ✓
  1139 V3     abstract  AI        DL             8.97     22m ✗
  1140 V3     abstract  Unclear   AI             9.07     22m ✗
  1141 V3     abstract  Unclear   AI             9.09     22m ✗
  1142 V3     abstract  Unclear   ML             8.97     21m ✗
  1143 V3     abstract  Unclear   AI             9.02     21m ✗
  1144 V3     abstract  ML        ML             9.07     21m ✓
  transient error — retry 1/6 in 4s


  1145 V3     abstract  AI        DL            13.46     20m ✗
  1146 V3     abstract  Unclear   ML             8.97     20m ✗
  1147 V3     abstract  ML        ML             9.05     19m ✓
  1148 V3     abstract  AI        Unclear        8.89     19m ✗
  1149 V3     abstract  ML        Unclear        9.45     19m ✗
  transient error — retry 1/6 in 4s


  transient error — retry 2/6 in 8s


  1150 V3     abstract  AI        DL           133.63     18m ✗
  1151 V3     abstract  Unclear   DL            60.09     18m ✗
  1152 V3     abstract  AI        AI            59.45     18m ✓
  1153 V3     abstract  ML        DL            59.72     17m ✗
  1154 V3     abstract  Unclear   DL            59.33     17m ✗
  1155 V3     abstract  Unclear   ML            61.31     17m ✗
  1156 V3     abstract  ML        ML            58.85     16m ✓
  1157 V3     abstract  Unclear   ML            59.67     16m ✗
  1158 V3     abstract  ML        ML            60.36     16m ✓
  1159 V3     abstract  Unclear   DL            59.55     15m ✗
  1160 V3     abstract  Unclear   AI            59.67     15m ✗
  1161 V3     abstract  Unclear   AI            60.02     15m ✗
  1162 V3     abstract  Unclear   AI            60.18     14m ✗
  1163 V3     abstract  Unclear   DL            59.02     14m ✗
  1164 V3     abstract  ML        ML            59.60     14m ✓
  1165 V3     abstract  Unclear   ML    

In [6]:
# ============================================================
# RETRY PASS — recover rows that exhausted their retries during the run
# ============================================================
results = pd.read_csv(OUTFILE)
failed = results[results.predicted_label == "ERROR"]
print(f"Retrying {len(failed)} failed rows...", flush=True)

fixed = 0
for i, row in failed.iterrows():
    src = df.loc[int(row["paper_id"])]
    text_block, _ = build_input(src)
    if text_block is None:
        continue

    t0 = time.time()
    raw, err = call_model(PROMPTS[row["prompt_version"]].format(text=text_block), MODEL)
    if raw:
        results.loc[i, ["predicted_label", "raw_output", "elapsed_sec", "error"]] = [
            extract_label(raw, row["prompt_version"]), raw, round(time.time()-t0, 2), ""
        ]
        fixed += 1
    time.sleep(2)

results.to_csv(OUTFILE, index=False)
print(f"Recovered {fixed}/{len(failed)} rows", flush=True)

Retrying 13 failed rows...
Recovered 13/13 rows


In [7]:
results = pd.read_csv(OUTFILE)

results["correct"] = (
    results["predicted_label"].map(normalise)
    == results["manual_label"].map(normalise)
)

summary = results.groupby("prompt_version").agg(
    n=("correct", "size"),
    accuracy=("correct", "mean"),
    unparseable=("predicted_label", lambda s: (s == "UNPARSEABLE").sum()),
    mean_sec=("elapsed_sec", "mean"),
).round(3)

print(summary)
print(f"\nErrors: {(results['error'].astype(str) != '').sum()}")

from google.colab import files
files.download(OUTFILE)

                  n  accuracy  unparseable  mean_sec
prompt_version                                      
V1              300     0.383            0    20.242
V2.1            300     0.397            0    16.845
V2.2            300     0.407            0    27.324
V3              300     0.343            0    28.996

Errors: 1200


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Cell D — Quick check, then download

In [8]:
results = pd.read_csv(OUTFILE)

results["correct"] = (
    results["predicted_label"].map(normalise)
    == results["manual_label"].map(normalise)
)

summary = results.groupby("prompt_version").agg(
    n=("correct", "size"),
    accuracy=("correct", "mean"),
    unparseable=("predicted_label", lambda s: (s == "UNPARSEABLE").sum()),
    mean_sec=("elapsed_sec", "mean"),
).round(3)

print(summary)
print(f"\nErrors: {(results['error'].astype(str) != '').sum()}")

from google.colab import files
files.download(OUTFILE)

                  n  accuracy  unparseable  mean_sec
prompt_version                                      
V1              300     0.383            0    20.242
V2.1            300     0.397            0    16.845
V2.2            300     0.407            0    27.324
V3              300     0.343            0    28.996

Errors: 1200


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
print(df[MANUAL_COL].value_counts())

Manual Classification
Unclear                    106
Machine Learning            99
Artificial Intelligence     52
Deep Learning               29
Name: count, dtype: int64


In [10]:
print(results)

      paper_id       model_version prompt_version input_source manual_label  \
0            0  gemma-4-26b-a4b-it             V1     abstract           DL   
1            1  gemma-4-26b-a4b-it             V1     abstract           ML   
2            2  gemma-4-26b-a4b-it             V1     abstract           DL   
3            3  gemma-4-26b-a4b-it             V1     abstract           AI   
4            4  gemma-4-26b-a4b-it             V1     abstract           DL   
...        ...                 ...            ...          ...          ...   
1195       295  gemma-4-26b-a4b-it             V3     abstract      Unclear   
1196       296  gemma-4-26b-a4b-it             V3     abstract           AI   
1197       297  gemma-4-26b-a4b-it             V3     abstract           ML   
1198       298  gemma-4-26b-a4b-it             V3     abstract      Unclear   
1199       299  gemma-4-26b-a4b-it             V3     abstract           AI   

     predicted_label                               